# ARCHILLES Embedding Server (Colab)

GPU-gestuetztes BGE-M3 Embedding fuer vorbereitete JSONL-Chunks.

**Vorbereitung:**
1. Laufzeit > Laufzeittyp aendern > **T4 GPU** auswaehlen
2. Alle Zellen der Reihe nach ausfuehren
3. Die angezeigte URL lokal verwenden

In [ ]:
# --- Zelle 1: Dependencies installieren + GPU pruefen ---
!pip install -q sentence-transformers fastapi uvicorn

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name} ({vram:.1f} GB VRAM)')
else:
    print('WARNUNG: Keine GPU! Laufzeit > Laufzeittyp aendern > T4 GPU')

In [ ]:
# --- Zelle 2: Embedding Server schreiben ---
%%writefile embedding_server.py
import gzip as gzip_module
import json
import os
import time
import logging

from fastapi import FastAPI, Header, HTTPException, Request
from starlette.middleware.gzip import GZipMiddleware
from typing import Optional

logger = logging.getLogger(__name__)
API_TOKEN = os.environ.get('EMBEDDING_API_TOKEN')

app = FastAPI(title='ARCHILLES Embedding Server')
app.add_middleware(GZipMiddleware, minimum_size=1000)

_model = None
_model_name = 'BAAI/bge-m3'
_dimension = 1024

def _get_model():
    global _model
    if _model is None:
        from sentence_transformers import SentenceTransformer
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        logger.info(f'Loading {_model_name} on {device}...')
        _model = SentenceTransformer(_model_name, device=device)
        logger.info('Model loaded.')
    return _model

def _check_auth(authorization):
    if not API_TOKEN:
        return
    if not authorization or not authorization.startswith('Bearer '):
        raise HTTPException(status_code=401, detail='Missing token')
    if authorization[7:] != API_TOKEN:
        raise HTTPException(status_code=403, detail='Invalid token')

@app.get('/health')
def health():
    import torch
    return {'model': _model_name, 'dimension': _dimension,
            'device': 'cuda' if torch.cuda.is_available() else 'cpu',
            'ready': _model is not None}

@app.post('/embed')
async def embed(request: Request, authorization: Optional[str] = Header(None)):
    _check_auth(authorization)
    body = await request.body()
    if request.headers.get('content-encoding') == 'gzip':
        body = gzip_module.decompress(body)
    data = json.loads(body)
    texts = data.get('texts', [])
    if not texts:
        return {'embeddings': [], 'model': _model_name, 'dimension': _dimension,
                'count': 0, 'duration_seconds': 0.0}
    if len(texts) > 10000:
        raise HTTPException(status_code=400, detail='Max 10000 texts per request')
    model = _get_model()
    start = time.time()
    vectors = model.encode(texts, batch_size=64, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)
    duration = time.time() - start
    return {'embeddings': vectors.tolist(), 'model': _model_name,
            'dimension': _dimension, 'count': len(texts),
            'duration_seconds': round(duration, 3)}

@app.on_event('startup')
def startup():
    logger.info('Pre-loading model...')
    _get_model()
    logger.info('Server ready.')

In [ ]:
# --- Zelle 3: Server starten + auf Bereitschaft warten ---
import subprocess, time, json, urllib.request

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'embedding_server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('Server startet... (BGE-M3 wird geladen, ~1-2 Min.)')
ready = False
for i in range(90):
    time.sleep(2)
    try:
        resp = urllib.request.urlopen('http://localhost:8000/health', timeout=5)
        data = json.loads(resp.read())
        if data.get('ready'):
            print(f"\nServer bereit! Device: {data['device']}, Model: {data['model']}")
            ready = True
            break
    except Exception:
        print('.', end='', flush=True)

if not ready:
    print('\nTimeout! Server konnte nicht gestartet werden.')
    print('Pruefe: Laufzeit > Laufzeittyp aendern > T4 GPU ausgewaehlt?')

In [ ]:
# --- Zelle 4: Cloudflare-Tunnel starten ---
import subprocess, threading, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared

public_url = [None]
url_found = threading.Event()

def watch_tunnel(proc):
    for line in iter(proc.stderr.readline, b''):
        text = line.decode(errors='replace')
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', text)
        if m:
            public_url[0] = m.group(1)
            url_found.set()

tunnel = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
threading.Thread(target=watch_tunnel, args=(tunnel,), daemon=True).start()

url_found.wait(timeout=30)
if public_url[0]:
    print(f"{'='*60}")
    print(f"EMBEDDING SERVER URL: {public_url[0]}")
    print(f"{'='*60}")
    print(f"\nLokal ausfuehren:")
    print(f"  python scripts/rag_demo.py embed \\")
    print(f"    --mode remote \\")
    print(f"    --host {public_url[0]} --port 443 \\")
    print(f"    --input-dir ./prepared_chunks_test")
else:
    print('Tunnel konnte nicht gestartet werden. Nochmal ausfuehren.')

In [ ]:
# --- Zelle 5: Health-Check ueber Tunnel (optional) ---
if public_url[0]:
    import urllib.request, json
    resp = urllib.request.urlopen(f"{public_url[0]}/health", timeout=10)
    data = json.loads(resp.read())
    print(f"Remote Health-Check: {data}")

## Lokale Ausfuehrung

### 1. Test-Subset erstellen (lokal, z.B. 50 Buecher)

```bash
mkdir prepared_chunks_test
ls prepared_chunks/*.jsonl | head -50 | while read f; do cp "$f" prepared_chunks_test/; done
```

### 2. Embedding starten

```bash
python scripts/rag_demo.py embed \
  --mode remote \
  --host <URL_VON_OBEN> --port 443 \
  --input-dir ./prepared_chunks_test
```

### Hinweise

- **Resume:** Bei Abbruch einfach nochmal starten, `.embed_checkpoint.json` trackt den Fortschritt
- **Session-Limit:** Colab Free trennt nach ~12h, Colab Pro nach ~24h
- **Bandbreite:** ~250 KB/Buch, 50 Buecher ~ 12 MB gesamt
- **Geschwindigkeit:** T4 GPU ~ 20-30s pro Buch
- **Fuer den vollen Bestand** (8.254 Buecher) empfiehlt sich Vast.ai/RunPod statt Colab

In [ ]:
# --- Zelle 6: Server am Laufen halten ---
# Diese Zelle blockiert, damit die Colab-Session nicht einschlaeft.
# Manuell stoppen wenn fertig (Stop-Button oder Strg+M I).
import time
print('Server laeuft. Diese Zelle offen lassen.')
print('Stoppen: Stop-Button oder Laufzeit > Alle Ausfuehrungen unterbrechen')
try:
    while True:
        time.sleep(60)
        # Quick health check
        try:
            resp = urllib.request.urlopen('http://localhost:8000/health', timeout=5)
            data = json.loads(resp.read())
            print(f"[{time.strftime('%H:%M')}] Server OK, device={data['device']}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M')}] Health-Check fehlgeschlagen: {e}")
except KeyboardInterrupt:
    print('\nGestoppt.')